### Notebook 范围

### 目的
用 BWCN reference O3 评估 0003、0008、0013、0014、0019 的事件强度。

### 科学问题
0008 是否仍然是最严重的 spring Arctic O3 depletion 个例？

### 方法
读取 BWCN_partial_O3_all_ranges.nc，使用 60-90N、30-70 hPa partial O3，比较 Jan1-May30 演变和 Mar-Apr 最小值。

### 输出
outputs/figures/01_reference 和 outputs/tables/01_reference。

### 解读
最低的 Mar-Apr O3 minimum 对应最强臭氧损耗；这解释为什么 0008 最适合做 detailed source diagnosis。

### 注意
0003 hindcast 来自特殊分支，这里只用 BWCN .002 作为统一参考基线。


### 导入与公共路径

### 目的
加载 Extention_analysis 的公共函数、数据路径、输出目录和绘图参数。

### 科学问题
保证所有 notebook 使用同一套变量定义，避免 EPFlux、O3、U、Z300 的窗口和符号约定互相漂移。

### 方法
使用 hindcast_extension_utils.py 中的 DATA_ROOT、HINDCAST_ROOT、BWCN_ROOT、B2000_ROOT、FIG_DIR、TAB_DIR、CACHE_DIR 和 LOG_DIR。

### 输出
无图；只打印工作目录和 Hindcast 数据目录。

### 解读
如果路径打印正确，后续代码块会使用同一套 cleaned hindcast 产品。

### 注意
所有写入都限制在 code_cleaned/Hindcast_experiment/Extention_analysis/outputs 下；不会修改原始数据。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 绘制 reference O3 severity

### 目的
绘制 selected years 的参考 O3 演变和 Mar-Apr minimum。

### 科学问题
其他年份是否只是 0008 的重复，还是更适合作为稳健性/反馈检验？

### 方法
Panel A 为每日 O3 partial column；Panel B 为 Mar-Apr minimum。x 轴使用月份坐标。

### 输出
01_reference_O3_severity_selected_years.png/pdf 和 CSV 表。

### 解读
0008 若显著最低，说明它是最强事件；其他年份更适合检验机制链条哪些环节稳健。

### 注意
只使用 reference，不涉及 hindcast spread。


In [ ]:
fig_dir = figure_dir("01_reference")
tab_dir = table_dir("01_reference")
rows = []
ts_rows = []
for year in SELECTED_YEARS:
    ref, date = load_bwcn_reference_o3(year)
    if ref is None:
        continue
    doy = case_time_doy(f"{year}-01", date)
    mask = (doy >= 1) & (doy <= 151)
    ref = ref.isel(lead_time=mask)
    doy = doy[mask]
    vals = np.asarray(ref.values, dtype=float)
    ma = (doy >= mmdd_to_doy(3, 1)) & (doy <= mmdd_to_doy(4, 30))
    rows.append({"year": year, "MA_min_O3_DU": float(np.nanmin(vals[ma])), "Jan_May_mean_O3_DU": float(np.nanmean(vals))})
    for d, v in zip(doy, vals):
        ts_rows.append({"year": year, "doy": int(d), "O3_DU": float(v)})
sev = pd.DataFrame(rows)
ts = pd.DataFrame(ts_rows)
sev_csv = tab_dir / "01_reference_O3_severity_selected_years.csv"
ts_csv = tab_dir / "01_reference_O3_timeseries_selected_years.csv"
sev.to_csv(sev_csv, index=False); ts.to_csv(ts_csv, index=False)
sev.to_csv(TAB_DIR / "01_reference_O3_severity_selected_years.csv", index=False)
ts.to_csv(TAB_DIR / "01_reference_O3_timeseries_selected_years.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), constrained_layout=True)
for year, sub in ts.groupby("year"):
    lw = 2.8 if year == "0008" else 1.8
    axes[0].plot(sub["doy"], sub["O3_DU"], label=year, lw=lw)
axes[0].set_ylabel("O3 partial column (DU)\n60-90N, 30-70 hPa")
axes[0].set_title("BWCN reference O3 evolution")
set_month_axis(axes[0], 1, 151)
axes[0].legend(ncol=2, fontsize=9)
colors = ["tab:red" if y == "0008" else "0.55" for y in sev["year"]]
axes[1].bar(sev["year"], sev["MA_min_O3_DU"], color=colors, edgecolor="black")
axes[1].set_ylabel("Mar-Apr minimum O3 (DU)")
axes[1].set_title("Reference event severity")
axes[1].grid(axis="y", alpha=0.3)
savefig(fig, "01_reference_O3_severity_selected_years", fig_dir=fig_dir, notebook="01_reference_event_severity.ipynb", scientific_question="0008 是否是 selected years 中最强 O3 depletion 事件？", variables_windows="BWCN reference O3; 60-90N; 30-70 hPa; Jan1-May30 and Mar-Apr minimum", interpretation="Mar-Apr minimum 越低表示事件越强。", caveat="0003 是特殊分支，reference 只作为统一基线。", csv_table=sev_csv)
plt.show()
write_figure_guide()
